<a href="https://colab.research.google.com/github/sabrinaangel/air-quality-prediction-ML/blob/main/notebooks/air_quality_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌤️ Air Quality Classification Using Meteorological Parameters

**Author:** Sabrina Angel — Teknik Informatika, Universitas Dian Nuswantoro

**Course:** Pembelajaran Mesin (Machine Learning) — UAS Genap 2025/2026

### Problem Statement
Air pollution is a growing concern in many urban and industrial areas, and monitoring it typically relies on direct pollutant sensors (PM2.5, PM10, NO2, SO2, CO), which are expensive to install and maintain at scale. This project explores an alternative approach: **can we classify air quality using only meteorological and environmental context features** — Temperature, Humidity, Proximity to Industrial Areas, and Population Density — **without relying on direct pollutant readings**?

This is a realistic and practically useful problem, since meteorological stations are far more common and cheaper to deploy than dedicated pollutant sensors. A model that can reasonably estimate air quality category from weather and demographic context alone could serve as a low-cost, early-warning proxy in areas without dense pollutant-monitoring infrastructure.

**Business / Analytical Goal:** Build a multi-class classification model that predicts the `Air Quality` category (`Good`, `Moderate`, `Poor`, `Hazardous`) using only meteorological and contextual features, and package it into an interactive dashboard that non-technical stakeholders (e.g. local government, environmental agencies) can use to explore predictions and insights.

**Success Metric:** Since the target classes are imbalanced (see EDA below), overall accuracy alone is not a sufficient success measure. The project targets a model with **macro-averaged F1-score above 0.80** and **ROC-AUC above 0.90**, with particular attention to recall on the minority `Hazardous` class — because failing to detect hazardous air quality has a higher real-world cost than a false alarm.

### Dataset
- **Source:** Kaggle — [`Air Quality and Pollution Assessment`](https://www.kaggle.com/datasets/mujtabamatin/air-quality-and-pollution-assessment) by Mujtaba Matin
- **Size:** 5,000 rows × 10 columns
- **Features (raw):** Temperature, Humidity, PM2.5, PM10, NO2, SO2, CO, Proximity_to_Industrial_Areas, Population_Density
- **Target:** `Air Quality` — 4 categories (`Good`, `Moderate`, `Poor`, `Hazardous`)

## ✂️ Step 1: Data Loading & Feature Selection
We load the raw dataset and deliberately remove the direct pollutant columns (`PM2.5`, `PM10`, `NO2`, `SO2`, `CO`).

**Why remove them?** This is a feature selection decision, not an oversight. Pollutant concentrations are used *directly* to derive the `Air Quality` label in the original dataset, so including them would cause severe **data leakage** — the model would essentially be told the answer. By restricting the feature set to meteorological and contextual variables only (`Temperature`, `Humidity`, `Proximity_to_Industrial_Areas`, `Population_Density`), the resulting model reflects the realistic, harder problem stated above: predicting air quality *without* pollutant sensors.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('updated_pollution_dataset.csv')

# Preview the first 5 rows
df.head()

In [ ]:
# 1. Specify the pollutant columns to remove (feature selection to avoid data leakage)
pollutant_columns = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO']

# 2. Drop the pollutant columns from the dataset
df_features = df.drop(columns=pollutant_columns)

# 3. Preview the resulting feature set
df_features.head()

The `df_features` DataFrame now contains only the meteorological/contextual predictors plus the target column, and will be used for the remainder of the pipeline: `Temperature`, `Humidity`, `Proximity_to_Industrial_Areas`, `Population_Density`, `Air Quality`.

## 🔎 Step 2: Exploratory Data Analysis (EDA)
Before modeling, we explore `df_features` to understand data quality, the distribution of each feature, relationships between features, and patterns relative to the target `Air Quality`.

### 2.1 Data Quality Check
Checking for missing values, duplicate rows, and data types to confirm the dataset is ready for analysis.

In [ ]:
# 1. General dataset info (data types & non-null counts)
df_features.info()

# 2. Missing values per column
print("\nMissing values per column:")
print(df_features.isnull().sum())

# 3. Duplicate rows
print(f"\nNumber of duplicate rows: {df_features.duplicated().sum()}")

> ✅ **Result**: The dataset has 5,000 rows with **no missing values** and **no duplicate rows**, so no imputation or row removal is needed on that front. All features already have appropriate numeric data types.

### 2.2 Data Consistency Check
Beyond missing values and duplicates, we also check whether feature values make physical sense — this is a step that's easy to skip but important.

In [ ]:
# Relative humidity should logically fall within 0-100%.
# Let's check whether that assumption holds in this dataset.
invalid_humidity = df_features[df_features['Humidity'] > 100]
print(f"Number of rows with Humidity > 100%: {len(invalid_humidity)}")
print(f"Humidity range before fix: {df_features['Humidity'].min()} - {df_features['Humidity'].max()}")

> ⚠️ **Inconsistency found**: 195 rows (≈3.9% of the data) have `Humidity` values above 100%, up to 128.1%. Since relative humidity is physically bounded at 100%, these values are inconsistent — most likely sensor noise or a calibration artifact rather than a meaningful signal.
>
> **Decision**: instead of dropping these rows (which would discard ~4% of the dataset, including data for other perfectly valid features) or leaving physically impossible values in the training data, we **cap `Humidity` at 100%**. This preserves all 5,000 rows while removing the invalid signal.

In [ ]:
# Cap Humidity at a physically valid maximum of 100%
df_features['Humidity'] = df_features['Humidity'].clip(upper=100)

print(f"Humidity range after fix: {df_features['Humidity'].min()} - {df_features['Humidity'].max()}")
print(f"Remaining rows above 100%: {(df_features['Humidity'] > 100).sum()}")

### 2.3 Descriptive Statistics
Overall size, range, and spread (mean, std, min, max) of each numeric feature.

In [ ]:
# Descriptive statistics for all numeric features
df_features.describe().T

### 2.4 Target Class Distribution
Checking the proportion of each `Air Quality` category to see whether the dataset is balanced.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

class_order = ['Good', 'Moderate', 'Poor', 'Hazardous']

# 1. Count and percentage per class
class_counts = df_features['Air Quality'].value_counts().reindex(class_order)
class_pct = (class_counts / len(df_features) * 100).round(2)
print("Air Quality class distribution:")
print(pd.DataFrame({'Count': class_counts, 'Percentage (%)': class_pct}))

# 2. Visualize class distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df_features, x='Air Quality', order=class_order, palette='Blues_d')
plt.title('Air Quality Class Distribution')
plt.xlabel('Air Quality Category')
plt.ylabel('Number of Samples')
plt.show()

> 📌 **Insight 1 — Class Imbalance**: The classes are imbalanced — `Good` (2,000 / 40%) and `Moderate` (1,500 / 30%) dominate, while `Hazardous` has only 500 samples (10%). This matters for evaluation: accuracy alone can be misleading, so we rely on **Stratified K-Fold** (Step 6) and **per-class classification reports** (Step 8) to make sure performance on the minority `Hazardous` class stays visible.

### 2.5 Univariate Analysis — Feature Distributions
Examining the shape of each numeric feature's distribution using histograms with KDE.

In [ ]:
numeric_features = ['Temperature', 'Humidity', 'Proximity_to_Industrial_Areas', 'Population_Density']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.histplot(df_features[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

> 📌 **Insight 2 — Distribution Shapes**: `Temperature` and `Humidity` are roughly bell-shaped (slightly right-skewed), while `Proximity_to_Industrial_Areas` and `Population_Density` are more spread out. No extreme skew is present, so no special transformation (e.g. log-transform) is needed.

### 2.6 Outlier Detection
Using boxplots to check for extreme values in each feature.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, col in enumerate(numeric_features):
    sns.boxplot(y=df_features[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

> 📌 **Insight 3 — Outliers**: There are mild outliers in `Temperature` and `Humidity` (points above the upper whisker), but they are few and represent plausible extreme weather conditions rather than data entry errors. These are **kept** (not removed), since extreme weather is exactly the kind of condition that may be relevant to predicting the `Hazardous` class.

### 2.7 Multivariate Analysis — Feature Correlation
Examining linear relationships between features using a correlation heatmap.

In [ ]:
plt.figure(figsize=(6, 5))
corr = df_features[numeric_features].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap - Meteorological & Environmental Features')
plt.show()

> 📌 **Insight 4 — Feature Correlation**: Correlations between features are low to moderate (none close to ±1), meaning there is no serious multicollinearity issue. This is favorable for models like Logistic Regression, which are sensitive to highly correlated inputs.

### 2.8 Multivariate Analysis — Features vs. Target
Examining how each feature's distribution differs across `Air Quality` categories.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.boxplot(data=df_features, x='Air Quality', y=col, order=class_order, ax=axes[i], palette='Blues')
    axes[i].set_title(f'{col} by Air Quality')

plt.tight_layout()
plt.show()

> 📌 **Insight 5 — Feature Patterns by Class**: `Hazardous` and `Poor` samples tend to have lower `Proximity_to_Industrial_Areas` (i.e. closer to industrial zones) and higher `Temperature` compared to `Good`. This pattern is later confirmed by the SHAP results in Step 9, where both features turn out to be the most important predictors.

### 📝 Summary of Key EDA Insights
1. **Class Imbalance** — The `Hazardous` class (10%) is far smaller than `Good` (40%), so evaluation must rely on per-class metrics, not just overall accuracy.
2. **Data Inconsistency Fixed** — 195 rows had `Humidity` above the physically valid 100% maximum; these were capped rather than dropped.
3. **Feature Distributions** — Temperature and Humidity are roughly normal; no special transformation is required.
4. **Outliers Kept** — Mild outliers in Temperature/Humidity are retained, as they represent plausible extreme-weather conditions relevant to the `Hazardous` class.
5. **No Serious Multicollinearity** — Correlations between features are low-to-moderate, safe to use together in both Logistic Regression and Random Forest.
6. **Proximity & Temperature Most Discriminative** — These two features show the clearest separation across `Air Quality` classes, consistent with the SHAP feature importance results later in the notebook.

## 🧹 Step 3: Preprocessing Summary
Before splitting and modeling, here is a summary of the preprocessing decisions made so far and what remains:

- **Missing values**: none found — no imputation needed.
- **Duplicates**: none found — no rows removed.
- **Inconsistent values**: `Humidity` values above 100% were capped (Step 2.2).
- **Feature selection**: pollutant columns were removed to avoid data leakage (Step 1); the remaining 4 features are used.
- **Encoding**: the target `Air Quality` is a categorical string column. Scikit-learn's classifiers (`LogisticRegression`, `RandomForestClassifier`) can consume string class labels directly, so no manual label encoding is required for this pipeline.
- **Scaling**: still needed for `Logistic Regression` (gradient-based, scale-sensitive) but not for `Random Forest` (tree-based, scale-invariant) — this is applied per-model in the steps below, fitted only on the training set to avoid leakage.

## ✂️ Step 4: Train / Validation / Test Split
We split the data into **70% training, 15% validation, and 15% test**, using **stratified sampling** so that the class proportions (Step 2.4) are preserved in every split.

- **Training set**: used to fit models and run hyperparameter tuning (Step 6).
- **Validation set**: used to sanity-check tuned models before committing to a final choice.
- **Test set**: held out completely and touched only once, in Step 8, to report final, unbiased performance.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Separate features (X) and target (y)
X = df_features.drop(columns=['Air Quality'])
y = df_features['Air Quality']

# 2. First split off the test set (15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# 3. Split the remaining 85% into train (70% of total) and validation (15% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, stratify=y_trainval, random_state=42
)  # 0.1765 * 0.85 ≈ 0.15 of the total data

# 4. Confirm the split sizes
print(f"Training set:   {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")

## 🤖 Step 5: Baseline Model Training
We train two baseline models with default hyperparameters, evaluated on the **validation set**, to establish a reference point before tuning.

> 📌 **Note on Feature Scaling**:
> * **Logistic Regression**: requires `StandardScaler` because it relies on gradient-based optimization, where differences in feature scale can hurt convergence.
> * **Random Forest**: does not require feature scaling, because it is a tree-based model that evaluates split conditions on individual features independently of their scale.

**Baseline model 1: Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Fit the scaler on the training set only, then transform train/val/test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 2. Train baseline Logistic Regression
baseline_lr = LogisticRegression(max_iter=1000, random_state=42)
baseline_lr.fit(X_train_scaled, y_train)

# 3. Evaluate on the validation set
val_pred_lr_baseline = baseline_lr.predict(X_val_scaled)
val_acc_lr_baseline = accuracy_score(y_val, val_pred_lr_baseline)
print(f"Baseline Logistic Regression — Validation Accuracy: {val_acc_lr_baseline * 100:.2f}%")

**Baseline model 2: Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Train baseline Random Forest (no scaling needed)
baseline_rf = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_rf.fit(X_train, y_train)

# 2. Evaluate on the validation set
val_pred_rf_baseline = baseline_rf.predict(X_val)
val_acc_rf_baseline = accuracy_score(y_val, val_pred_rf_baseline)
print(f"Baseline Random Forest — Validation Accuracy: {val_acc_rf_baseline * 100:.2f}%")

## 🎯 Step 6: Hyperparameter Tuning (GridSearchCV)
We tune both models using `GridSearchCV` with 5-fold cross-validation **within the training set only** (the validation and test sets are never used for tuning), then check the tuned models against the validation set to confirm they actually improved over the baselines.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Grid Search for Logistic Regression
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100], 'solver': ['lbfgs'], 'max_iter': [1000]}
grid_lr = GridSearchCV(LogisticRegression(random_state=42), param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train)

print("=== Logistic Regression ===")
print("Best params:", grid_lr.best_params_)
print(f"Best CV Accuracy (train set): {grid_lr.best_score_ * 100:.2f}%")

# 2. Grid Search for Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("\n=== Random Forest ===")
print("Best params:", grid_rf.best_params_)
print(f"Best CV Accuracy (train set): {grid_rf.best_score_ * 100:.2f}%")

# 3. Extract the best estimators from tuning
tuned_lr = grid_lr.best_estimator_
tuned_rf = grid_rf.best_estimator_

# 4. Confirm improvement on the validation set (tuned vs. baseline)
val_pred_lr_tuned = tuned_lr.predict(X_val_scaled)
val_pred_rf_tuned = tuned_rf.predict(X_val)

val_acc_lr_tuned = accuracy_score(y_val, val_pred_lr_tuned)
val_acc_rf_tuned = accuracy_score(y_val, val_pred_rf_tuned)

print(f"\nLogistic Regression — Validation Accuracy: {val_acc_lr_baseline*100:.2f}% (baseline) -> {val_acc_lr_tuned*100:.2f}% (tuned)")
print(f"Random Forest       — Validation Accuracy: {val_acc_rf_baseline*100:.2f}% (baseline) -> {val_acc_rf_tuned*100:.2f}% (tuned)")

## ⚖️ Step 7: Model Selection — 10-Fold Stratified Cross-Validation & Paired T-Test
To decide between the two tuned models more rigorously than a single validation-set comparison, we run a **10-Fold Stratified Cross-Validation** and a **Paired T-Test** ($\alpha = 0.05$).

This step uses **`X_trainval` / `y_trainval`** (training + validation combined) rather than the full dataset — the test set stays completely untouched until Step 8, so the final reported performance remains an unbiased estimate on truly unseen data.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from scipy import stats

# 1. Stratified 10-fold splitting scheme
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 2. Pipelines using each model's tuned hyperparameters, so scaling is refit per fold (no leakage)
pipe_lr = make_pipeline(StandardScaler(), LogisticRegression(**grid_lr.best_params_, random_state=42))
model_rf_cv = RandomForestClassifier(**grid_rf.best_params_, random_state=42)

# 3. Cross-validated accuracy scores on train+validation data
scores_lr = cross_val_score(pipe_lr, X_trainval, y_trainval, cv=kfold, scoring='accuracy')
scores_rf = cross_val_score(model_rf_cv, X_trainval, y_trainval, cv=kfold, scoring='accuracy')

# 4. Paired T-Test between the two score distributions
t_stat, p_value = stats.ttest_rel(scores_rf, scores_lr)

# 5. Report results
print(f"Mean CV Accuracy — Logistic Regression: {scores_lr.mean() * 100:.2f}%")
print(f"Mean CV Accuracy — Random Forest:       {scores_rf.mean() * 100:.2f}%")
print(f"Paired T-Test p-value: {p_value:.6f}")

if p_value < 0.05:
    print("CONCLUSION: The performance difference is statistically significant (p < 0.05).")
else:
    print("CONCLUSION: The performance difference is NOT statistically significant.")

> ✅ Random Forest consistently and significantly outperforms Logistic Regression, so it is selected as the **final model** for deployment and further interpretation.

## 📊 Step 8: Final Model Training & Test Set Evaluation
With Random Forest selected and its best hyperparameters known (from Step 6), we retrain **one final model on the combined train + validation data** (`X_trainval`, `y_trainval`) to make use of as much data as possible, then evaluate it **exactly once** on the held-out **test set** — data the model has never seen in any previous step.

We report Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and **ROC-AUC** (one-vs-rest, since this is a multi-class problem).

In [ ]:
# Train the final Random Forest on train+validation using the best hyperparameters found in Step 6
final_model = RandomForestClassifier(**grid_rf.best_params_, random_state=42)
final_model.fit(X_trainval, y_trainval)

# Predict on the untouched test set
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Final Random Forest — Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

# 1. Classification report (Precision, Recall, F1-Score per class)
print("=== CLASSIFICATION REPORT — Final Random Forest (Test Set) ===")
print(classification_report(y_test, y_test_pred))

# 2. ROC-AUC (multi-class, one-vs-rest)
roc_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
print(f"Macro-average ROC-AUC (One-vs-Rest): {roc_auc:.4f}")

# 3. Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred, labels=final_model.classes_)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=final_model.classes_, yticklabels=final_model.classes_)
plt.title('Confusion Matrix - Final Random Forest (Test Set)')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

> ✅ **Result**: The final Random Forest model, trained on train+validation and evaluated on a completely unseen test set, confirms strong and consistent performance across all four classes, including the minority `Hazardous` class — supporting the success criteria defined in the Problem Statement (macro F1 > 0.80, ROC-AUC > 0.90).

## 🔍 Step 9: Model Interpretation with SHAP
We use **SHAP (SHapley Additive exPlanations)** to interpret how each feature contributes to the final Random Forest model's predictions — a more rigorous, game-theoretically grounded alternative to the default `feature_importances_`.

In [ ]:
!pip install shap -q

In [ ]:
import shap

# 1. Build a tree-based explainer for the final model
explainer = shap.TreeExplainer(final_model)

# 2. Sample 200 rows from the test set to keep computation light
X_sample = X_test.sample(200, random_state=42)
shap_values = explainer.shap_values(X_sample)

# 3. Summary plot of feature contributions across all classes
shap.summary_plot(shap_values, X_sample, plot_type="bar", class_names=final_model.classes_)

### 💡 Feature Importance Interpretation
1. **Dominant factors**: `Proximity_to_Industrial_Areas` and `Temperature` show the highest SHAP importance scores.
2. **Secondary factors**: `Humidity` and `Population_Density` also contribute meaningfully, helping capture non-linear patterns that the model uses to separate classes.
3. **Conclusion**: In the absence of direct pollutant sensor data, industrial proximity and ambient temperature act as the strongest available proxies for air quality — consistent with the EDA findings in Step 2.8.

## 💾 Step 10: Save Model & Scaler Artifacts
Exporting the final trained Random Forest model and the `StandardScaler` (used for the Logistic Regression comparison / potential future use) with `joblib`, for deployment in the Streamlit app.

In [ ]:
import joblib

# Save the final Random Forest model and the scaler
joblib.dump(final_model, 'rf_air_quality_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model and scaler saved successfully!")

# If running in Google Colab, download automatically:
try:
    from google.colab import files
    files.download('rf_air_quality_model.pkl')
    files.download('scaler.pkl')
except ImportError:
    pass  # Not running in Colab — files are already saved locally

## 📌 Final Conclusion

Based on the experiments conducted:
1. **Model Performance**: The **Random Forest** model outperforms Logistic Regression at classifying air quality categories from meteorological and contextual features alone.
2. **Statistical Validation**: A 10-Fold Stratified Cross-Validation and Paired T-Test confirm that Random Forest's superiority over Logistic Regression is statistically significant (p < 0.05).
3. **Unbiased Final Evaluation**: The final model was retrained on train+validation data and evaluated exactly once on a held-out test set, achieving strong Accuracy, macro F1-Score, and ROC-AUC across all four classes.
4. **Key Drivers**: According to the SHAP feature importance analysis, `Proximity_to_Industrial_Areas` and `Temperature` are the most influential predictors of air quality.
5. **Artifact Export**: The final Random Forest model and scaler have been exported to `.pkl` format for deployment in the accompanying Streamlit application.